# Chapter 22 Companion Notebook: Topic Modeling and Theme Discovery

**Book:** *Business Analytics and Artificial Intelligence: An Advanced Guide to Data-Driven Decision Making*  
**Book authors:** Hyunhwan "Aiden" Lee and Reo Song  
**Notebook author:** Hyunhwan Aiden Lee  
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/indy16mm/business-analytics-ai/blob/main/notebooks/Ch22_Topic_Modeling_and_Theme_Discovery.ipynb)

This notebook accompanies Chapter 22 of the book.

**License:** Use of this notebook is governed by the repository's
[Limited Companion Materials License](../LICENSE).




This notebook is designed as a classroom appendix. It uses synthetic business text data so that every step can run safely in Google Colab without external files, paid APIs, or private customer information. The goal is not to turn topic modeling into a black box. The goal is to show how analysts move from raw customer language to candidate topics, business-facing themes, validation evidence, segment and time comparisons, and governance routines.

## Why this matters (business framing)

Marketing and service teams often have more text than they can read: reviews, survey verbatims, support tickets, community posts, chat transcripts, and escalation notes. Topic modeling helps reduce this overload by surfacing recurring language patterns. In business analytics, however, the model output is only the starting point. A topic becomes useful only when it is interpreted, named, validated, and connected to a decision.

This notebook treats topic modeling as *theme discovery*. We will build interpretable baselines, compare count-based and dense representations, fit matrix factorization and probabilistic topic models, create a lightweight embedding-clustering workflow, validate theme quality, and turn themes into dashboards and monitoring signals. Along the way, we will keep the chapter's core distinction visible: a topic is a model output, while a theme is a business-facing interpretation.

## Agenda

1. Setup and reproducibility
2. Synthetic business text corpus with metadata
3. Unit of text and corpus quality checks
4. Interpretable keyword and phrase baselines
5. Representations for theme discovery
6. Topic modeling with nonnegative matrix factorization
7. Probabilistic topic modeling with LDA
8. Semantic-style topic discovery with dense vectors and clustering
9. Validation: coherence, distinctness, stability, and theme cards
10. Theme comparison across segments and time
11. Drift monitoring and governance outputs
12. Exercises

## Connection map

Chapter 20 focused on text preprocessing and data quality. Chapter 21 introduced embeddings and similarity. Chapter 22 uses those ideas to discover themes in a corpus. The workflow in this notebook is deliberately practical: define the document unit, prepare a comparable corpus, choose a representation, discover candidate topics, review evidence, assign stable theme names, compare prevalence across business groups, and monitor drift over time.

Later chapters on sentiment, transformer-based NLP, and retrieval-augmented text mining can reuse the theme catalog created here. A stable theme catalog can become a supervised label set, a dashboard dimension, a retrieval filter, or an audit artifact.

In [ ]:
# ============================================================
# 1. Setup and reproducibility
# - install missing packages if needed
# - import libraries
# - set seeds
# - configure output folders
# ============================================================

import os
# Keep classroom notebooks lightweight and prevent BLAS/OpenMP oversubscription in small CPU environments.
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("VECLIB_MAXIMUM_THREADS", "1")
os.environ.setdefault("NUMEXPR_NUM_THREADS", "1")

import importlib.util
import subprocess
import sys
from pathlib import Path

REQUIRED_PACKAGES = {
    "numpy": "numpy",
    "pandas": "pandas",
    "matplotlib": "matplotlib",
    "sklearn": "scikit-learn",
}

for import_name, pip_name in REQUIRED_PACKAGES.items():
    if importlib.util.find_spec(import_name) is None:
        print(f"Installing {pip_name}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name])

import json
import math
import random
import re
import warnings
from collections import Counter, defaultdict
from itertools import combinations

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display
from sklearn.cluster import KMeans
from sklearn.decomposition import LatentDirichletAllocation, NMF, PCA, TruncatedSVD
from sklearn.feature_extraction.text import CountVectorizer, ENGLISH_STOP_WORDS, TfidfVectorizer
from sklearn.metrics import adjusted_rand_score
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import normalize

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

OUTPUT_DIR = Path("/content/ch22_outputs") if Path("/content").exists() else Path("ch22_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_colwidth", 130)
pd.set_option("display.width", 160)

print("Setup complete")
print(f"Output directory: {OUTPUT_DIR.resolve()}")

## Utility functions

These helper functions keep the main sections focused on theme discovery and decision logic. They handle text cleaning, boilerplate removal, topic descriptors, representative documents, coherence screening, stability checks, and compact visualizations.

In [ ]:
# ============================================================
# Utility functions
# ============================================================

CUSTOM_STOP_WORDS = set(ENGLISH_STOP_WORDS).union({
    "customer", "customers", "product", "products", "service", "issue", "issues",
    "really", "just", "like", "said", "says", "want", "wanted", "need", "needed",
})

BOILERPLATE_PATTERNS = [
    r"thank you for contacting support",
    r"we value your feedback",
    r"this case has been routed",
    r"a representative will review your case",
]


def normalize_text(text):
    """Lowercase, remove most punctuation, and collapse whitespace."""
    text = str(text).lower()
    text = re.sub(r"[^a-z0-9\s'-]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def remove_boilerplate(text):
    """Remove synthetic support templates that can distort discovered themes."""
    cleaned = normalize_text(text)
    for pattern in BOILERPLATE_PATTERNS:
        cleaned = re.sub(pattern, " ", cleaned)
    cleaned = re.sub(r"\s+", " ", cleaned).strip()
    return cleaned


def simple_tokenize(text, stop_words=CUSTOM_STOP_WORDS, min_len=3):
    tokens = re.findall(r"[a-z][a-z']+", normalize_text(text))
    return [t for t in tokens if t not in stop_words and len(t) >= min_len]


def split_sentences(text):
    parts = re.split(r"(?<=[.!?])\s+", str(text).strip())
    return [p.strip() for p in parts if len(p.strip()) > 0]


def top_weighted_terms(matrix, feature_names, top_n=12):
    """Return the highest-weighted terms from a vector or component row."""
    arr = np.asarray(matrix).ravel()
    if arr.size == 0:
        return []
    top_idx = arr.argsort()[::-1][:top_n]
    return [(feature_names[i], float(arr[i])) for i in top_idx]


def component_terms_table(components, feature_names, model_name="Topic", top_n=10):
    rows = []
    for topic_id, component in enumerate(components):
        terms = [term for term, weight in top_weighted_terms(component, feature_names, top_n=top_n)]
        rows.append({
            "topic_id": topic_id,
            "model": model_name,
            "top_terms": ", ".join(terms),
        })
    return pd.DataFrame(rows)


def representative_docs(df, topic_weights, topic_id, score_name="weight", top_n=3):
    scores = topic_weights[:, topic_id]
    idx = np.argsort(scores)[::-1][:top_n]
    rows = df.iloc[idx][["doc_id", "channel", "segment", "month_label", "primary_theme", "text"]].copy()
    rows.insert(1, score_name, scores[idx].round(3))
    return rows


def plot_bar(series, title, ylabel, filename=None, rotation=25):
    ax = series.plot(kind="bar", figsize=(9, 4))
    ax.set_title(title)
    ax.set_ylabel(ylabel)
    ax.set_xlabel("")
    ax.tick_params(axis="x", rotation=rotation)
    plt.tight_layout()
    if filename:
        path = OUTPUT_DIR / filename
        plt.savefig(path, dpi=160, bbox_inches="tight")
        print(f"Saved figure: {path}")


def plot_heatmap(pivot, title, filename=None):
    fig, ax = plt.subplots(figsize=(10, 4.8))
    im = ax.imshow(pivot.values, aspect="auto")
    ax.set_title(title)
    ax.set_yticks(np.arange(len(pivot.index)))
    ax.set_yticklabels(pivot.index)
    ax.set_xticks(np.arange(len(pivot.columns)))
    ax.set_xticklabels(pivot.columns, rotation=45, ha="right")
    cbar = plt.colorbar(im, ax=ax)
    cbar.set_label("Prevalence")
    plt.tight_layout()
    if filename:
        path = OUTPUT_DIR / filename
        plt.savefig(path, dpi=160, bbox_inches="tight")
        print(f"Saved figure: {path}")


def topic_coherence_npmi(texts, topic_terms, min_df=2):
    """A lightweight NPMI-style screening metric based on document co-occurrence."""
    if len(topic_terms) < 2:
        return np.nan
    vectorizer = CountVectorizer(vocabulary=sorted(set(topic_terms)), binary=True)
    X = vectorizer.fit_transform(texts)
    if X.shape[1] < 2:
        return np.nan
    N = X.shape[0]
    arr = X.toarray().astype(bool)
    term_to_idx = {term: i for i, term in enumerate(vectorizer.get_feature_names_out())}
    scores = []
    eps = 1e-12
    for a, b in combinations(topic_terms, 2):
        if a not in term_to_idx or b not in term_to_idx:
            continue
        ia, ib = term_to_idx[a], term_to_idx[b]
        count_a = arr[:, ia].sum()
        count_b = arr[:, ib].sum()
        count_ab = np.logical_and(arr[:, ia], arr[:, ib]).sum()
        if count_a < min_df or count_b < min_df or count_ab == 0:
            continue
        p_a = count_a / N
        p_b = count_b / N
        p_ab = count_ab / N
        pmi = math.log((p_ab + eps) / (p_a * p_b + eps))
        npmi = pmi / (-math.log(p_ab + eps))
        scores.append(npmi)
    return float(np.mean(scores)) if scores else np.nan


def top_terms_as_lists(components, feature_names, top_n=10):
    result = []
    for component in components:
        result.append([term for term, weight in top_weighted_terms(component, feature_names, top_n=top_n)])
    return result


def mean_pairwise_jaccard(term_lists):
    scores = []
    for a, b in combinations(term_lists, 2):
        sa, sb = set(a), set(b)
        denom = len(sa.union(sb))
        if denom:
            scores.append(len(sa.intersection(sb)) / denom)
    return float(np.mean(scores)) if scores else np.nan


def best_match_stability(list_a, list_b):
    """Average best Jaccard match from topics in A to topics in B."""
    best_scores = []
    for a in list_a:
        sa = set(a)
        matches = []
        for b in list_b:
            sb = set(b)
            denom = len(sa.union(sb))
            matches.append(len(sa.intersection(sb)) / denom if denom else 0.0)
        best_scores.append(max(matches) if matches else 0.0)
    return float(np.mean(best_scores)) if best_scores else np.nan


def compact_text(text, width=165):
    text = re.sub(r"\s+", " ", str(text)).strip()
    return text if len(text) <= width else text[: width - 3] + "..."

## 2. Synthetic business text corpus with metadata

The corpus below mimics a customer feedback repository that combines app-store reviews, support tickets, survey verbatims, and community posts. Each record has a document unit, text, and metadata. The hidden `primary_theme` and `secondary_theme` columns are included only for classroom evaluation. In a real project, these labels would not be known at the discovery stage.

In [ ]:
# ============================================================
# 2. Synthetic business text corpus with metadata
# ============================================================

THEME_BLUEPRINTS = {
    "delivery_reliability": {
        "business_label": "Delivery reliability",
        "owner": "Operations and fulfillment",
        "terms": ["delivery", "late", "tracking", "shipping", "carrier", "arrival", "warehouse", "package", "rescheduled"],
        "sentences": [
            "The package arrived late and the tracking page did not match the delivery window.",
            "Shipping was delayed again, and the carrier rescheduled without a clear explanation.",
            "I like the order itself, but delivery reliability makes it hard to plan around the arrival.",
            "The warehouse update said shipped, while the package was still waiting for pickup.",
        ],
    },
    "pricing_fairness": {
        "business_label": "Pricing fairness",
        "owner": "Pricing and revenue management",
        "terms": ["price", "fee", "discount", "coupon", "charge", "expensive", "renewal", "promotion", "value"],
        "sentences": [
            "The renewal price felt higher than the value I received from the plan.",
            "The coupon disappeared at checkout, so the final charge felt unfair.",
            "I would stay longer if the price and fee structure were easier to understand.",
            "The promotion looked generous, but the total charge became expensive after added fees.",
        ],
    },
    "onboarding_friction": {
        "business_label": "Onboarding friction",
        "owner": "Growth and onboarding",
        "terms": ["setup", "activation", "tutorial", "login", "password", "instructions", "confusing", "profile", "steps"],
        "sentences": [
            "The setup steps were confusing and the activation email did not explain what to do next.",
            "I could not finish the tutorial because the login screen kept asking for a password reset.",
            "The instructions assume users already know the product, which makes onboarding frustrating.",
            "Creating the profile took too many steps before I could try the main feature.",
        ],
    },
    "app_stability": {
        "business_label": "App stability",
        "owner": "Product engineering",
        "terms": ["app", "crash", "freeze", "loading", "bug", "screen", "button", "update", "restart"],
        "sentences": [
            "The app freezes on the loading screen after the newest update.",
            "I tapped the checkout button twice and the screen crashed before the order was saved.",
            "The mobile experience is useful when it works, but the bug forces me to restart often.",
            "The app keeps crashing whenever I switch between the profile and payment screens.",
        ],
    },
    "service_response": {
        "business_label": "Service response",
        "owner": "Customer care",
        "terms": ["agent", "support", "chat", "callback", "hold", "representative", "response", "escalation", "waiting"],
        "sentences": [
            "The chat agent was polite, but the response did not solve the problem.",
            "I waited on hold and never received the promised callback from support.",
            "The representative kept escalating the case instead of giving a clear answer.",
            "Support was helpful once I reached a person, but the waiting time was too long.",
        ],
    },
    "product_quality": {
        "business_label": "Product quality",
        "owner": "Merchandising and product quality",
        "terms": ["quality", "material", "battery", "durable", "broken", "replacement", "defect", "packaging", "part"],
        "sentences": [
            "The material feels durable, but one replacement part arrived broken in the box.",
            "Battery quality is inconsistent, and the device loses charge faster than expected.",
            "The packaging looked premium, but the product had a visible defect when opened.",
            "I expected stronger quality control because the part failed after only a few uses.",
        ],
    },
    "competitor_comparison": {
        "business_label": "Competitor comparison",
        "owner": "Brand strategy",
        "terms": ["competitor", "alternative", "switch", "premium", "cheaper", "feature", "brand", "comparison", "market"],
        "sentences": [
            "A competitor offers the same feature for a cheaper monthly price.",
            "I may switch because the alternative brand feels more premium and easier to use.",
            "The market has better comparison options now, especially for small business customers.",
            "Your brand is still familiar, but the competitor has a simpler feature bundle.",
        ],
    },
}

CHANNELS = ["app_store_review", "support_ticket", "survey_verbatim", "community_post"]
SEGMENTS = ["New customer", "Loyal customer", "Price sensitive", "Premium subscriber", "Small business"]
PRODUCT_LINES = ["Mobile app", "Subscription service", "Smart home device"]
REGIONS = ["West", "South", "Northeast", "Midwest"]
MONTHS = pd.date_range("2025-01-01", periods=12, freq="MS")

CHANNEL_BIAS = {
    "app_store_review": {"app_stability": 4, "onboarding_friction": 2, "pricing_fairness": 1},
    "support_ticket": {"service_response": 4, "pricing_fairness": 2, "delivery_reliability": 2},
    "survey_verbatim": {"pricing_fairness": 2, "competitor_comparison": 2, "product_quality": 2},
    "community_post": {"competitor_comparison": 3, "product_quality": 2, "delivery_reliability": 1},
}

SEGMENT_BIAS = {
    "New customer": {"onboarding_friction": 4, "app_stability": 2},
    "Loyal customer": {"product_quality": 2, "service_response": 2},
    "Price sensitive": {"pricing_fairness": 5, "competitor_comparison": 2},
    "Premium subscriber": {"product_quality": 3, "pricing_fairness": 2},
    "Small business": {"delivery_reliability": 3, "service_response": 2, "competitor_comparison": 2},
}


def weighted_theme(channel, segment, month):
    weights = {theme: 1.0 for theme in THEME_BLUEPRINTS}
    for theme, weight in CHANNEL_BIAS[channel].items():
        weights[theme] += weight
    for theme, weight in SEGMENT_BIAS[segment].items():
        weights[theme] += weight
    if month.month in [10, 11, 12]:
        weights["delivery_reliability"] += 2.5
    if month.month in [1, 2]:
        weights["pricing_fairness"] += 1.5
    if month.month >= 9:
        weights["app_stability"] += 2.0
    themes, probs = zip(*weights.items())
    return random.choices(themes, weights=probs, k=1)[0]


def compose_document(primary, secondary=None, channel="survey_verbatim", month=None):
    primary_sentences = random.sample(THEME_BLUEPRINTS[primary]["sentences"], k=2)
    pieces = primary_sentences.copy()
    if secondary is not None:
        pieces.append(random.choice(THEME_BLUEPRINTS[secondary]["sentences"]))
    random.shuffle(pieces)
    text = " ".join(pieces)
    if month is not None and month.month >= 9 and primary == "app_stability":
        text += " After the Nova update, the AI assistant and voice recommendation feature sometimes freeze together."
    if month is not None and month.month >= 10 and primary == "service_response":
        text += " The chatbot automation made the support path feel longer before a real agent joined."
    if channel == "support_ticket" and random.random() < 0.28:
        text = "Thank you for contacting support. We value your feedback. This case has been routed to a representative. " + text
    return text

rows = []
for i in range(320):
    month = random.choice(list(MONTHS))
    channel = random.choice(CHANNELS)
    segment = random.choice(SEGMENTS)
    product_line = random.choice(PRODUCT_LINES)
    region = random.choice(REGIONS)
    primary = weighted_theme(channel, segment, month)
    secondary = None
    if random.random() < 0.28:
        secondary_candidates = [t for t in THEME_BLUEPRINTS if t != primary]
        secondary = random.choice(secondary_candidates)
    text = compose_document(primary, secondary, channel=channel, month=month)
    rows.append({
        "doc_id": f"D{i+1:04d}",
        "date": month + pd.Timedelta(days=random.randint(0, 27)),
        "month": month,
        "month_label": month.strftime("%Y-%m"),
        "channel": channel,
        "segment": segment,
        "product_line": product_line,
        "region": region,
        "customer_id": f"C{random.randint(1, 180):04d}",
        "primary_theme": primary,
        "secondary_theme": secondary if secondary is not None else "none",
        "text": text,
    })

# Add a small number of repeated or cross-posted records to make duplicate inflation visible.
duplicate_source_idx = random.sample(range(len(rows)), 12)
for j, idx in enumerate(duplicate_source_idx):
    source = rows[idx].copy()
    source["doc_id"] = f"DUP{j+1:03d}"
    source["channel"] = random.choice([c for c in CHANNELS if c != source["channel"]])
    source["date"] = source["date"] + pd.Timedelta(days=random.randint(1, 5))
    source["month"] = pd.Timestamp(source["date"].year, source["date"].month, 1)
    source["month_label"] = source["month"].strftime("%Y-%m")
    rows.append(source)

df = pd.DataFrame(rows).sort_values("date").reset_index(drop=True)
df["raw_text"] = df["text"]
df["clean_text"] = df["text"].apply(remove_boilerplate)
df["word_count"] = df["clean_text"].apply(lambda x: len(simple_tokenize(x)))
df["has_secondary_theme"] = df["secondary_theme"].ne("none")
df["contains_boilerplate"] = df["raw_text"].str.lower().str.contains("thank you for contacting support")

print(f"Documents: {len(df):,}")
print(f"Date range: {df['date'].min().date()} to {df['date'].max().date()}")
print(f"Documents with secondary themes: {df['has_secondary_theme'].sum():,}")
print(f"Documents with support boilerplate: {df['contains_boilerplate'].sum():,}")

display(df[["doc_id", "month_label", "channel", "segment", "primary_theme", "secondary_theme", "word_count", "text"]].head(8))

In [ ]:
# ============================================================
# Quick corpus view: theme, channel, and segment composition
# ============================================================

primary_theme_counts = df["primary_theme"].map(lambda x: THEME_BLUEPRINTS[x]["business_label"]).value_counts()
display(primary_theme_counts.rename("document_count").to_frame())
plot_bar(primary_theme_counts, "Synthetic corpus composition by hidden classroom theme", "Documents", "hidden_theme_distribution.png")

channel_segment_table = pd.crosstab(df["channel"], df["segment"])
display(channel_segment_table)

length_summary = df.groupby("channel")["word_count"].agg(["count", "mean", "median", "min", "max"]).round(1)
display(length_summary)

## 3. Unit of text and corpus quality checks

The unit of text is the unit of measurement. A review-level analysis answers a different question from a sentence-level analysis. A review can mix delivery, price, product quality, and service response. Splitting into smaller chunks can sharpen themes, but it can also remove context. The goal is not to make text as short as possible. The goal is to choose the unit that matches the decision.

In [ ]:
# ============================================================
# 3. Unit of text: document-level versus sentence-level views
# ============================================================

sentence_rows = []
for _, row in df.iterrows():
    for sent_id, sent in enumerate(split_sentences(row["raw_text"]), start=1):
        if len(simple_tokenize(sent)) >= 4:
            sentence_rows.append({
                "doc_id": row["doc_id"],
                "sentence_id": sent_id,
                "month_label": row["month_label"],
                "channel": row["channel"],
                "segment": row["segment"],
                "sentence_text": sent,
            })

sent_df = pd.DataFrame(sentence_rows)
print(f"Document-level rows: {len(df):,}")
print(f"Sentence-level rows after filtering very short sentences: {len(sent_df):,}")
print(f"Average sentences per document: {len(sent_df) / len(df):.2f}")

display(sent_df.head(8))

# Show a mixed document where sentence-level chunking can reveal different issues.
mixed_example = df[df["has_secondary_theme"]].sample(1, random_state=SEED).iloc[0]
print("Mixed document example")
print("Primary theme:", THEME_BLUEPRINTS[mixed_example["primary_theme"]]["business_label"])
print("Secondary theme:", THEME_BLUEPRINTS[mixed_example["secondary_theme"]]["business_label"])
print("\nFull document:")
print(compact_text(mixed_example["text"], width=500))
print("\nSentence chunks:")
for s in split_sentences(mixed_example["text"]):
    print("-", s)

In [ ]:
# ============================================================
# Quality checks that can distort discovered themes
# ============================================================

# Exact duplicates based on normalized raw text.
df["normalized_raw"] = df["raw_text"].apply(normalize_text)
duplicate_summary = (
    df.groupby("normalized_raw")
    .agg(documents=("doc_id", "count"), example_doc=("doc_id", "first"), example_text=("raw_text", "first"))
    .query("documents > 1")
    .sort_values("documents", ascending=False)
)

print(f"Exact duplicate groups: {len(duplicate_summary)}")
if len(duplicate_summary) > 0:
    display(duplicate_summary.head(5))

boilerplate_rate = df.groupby("channel")["contains_boilerplate"].mean().sort_values(ascending=False)
display((boilerplate_rate * 100).round(1).rename("boilerplate_percent").to_frame())

mix_by_month = pd.crosstab(df["month_label"], df["channel"], normalize="index").round(3)
display(mix_by_month.head())

print("Quality interpretation:")
print("Duplicates can inflate prevalence. Boilerplate can become a process topic. Channel mix can make trend changes look like customer-experience changes.")

## 4. Interpretable keyword and phrase baselines

Before fitting a topic model, it is useful to build a simple baseline. A baseline does not discover everything, but it creates a transparent measurement scaffold. It also helps stakeholders see how theme definitions, inclusion rules, and exclusion rules affect reported prevalence.

In [ ]:
# ============================================================
# 4. Keyword and phrase baseline
# ============================================================

BUSINESS_THEME_RULES = {
    blueprint["business_label"]: blueprint["terms"]
    for blueprint in THEME_BLUEPRINTS.values()
}

# Add a few phrases that are common in business language.
BUSINESS_THEME_RULES["Delivery reliability"] += ["delivery window", "tracking page", "shipping delayed"]
BUSINESS_THEME_RULES["Pricing fairness"] += ["renewal price", "final charge", "added fees"]
BUSINESS_THEME_RULES["Onboarding friction"] += ["password reset", "activation email", "setup steps"]
BUSINESS_THEME_RULES["App stability"] += ["loading screen", "keeps crashing", "newest update"]
BUSINESS_THEME_RULES["Service response"] += ["on hold", "promised callback", "real agent"]
BUSINESS_THEME_RULES["Product quality"] += ["quality control", "replacement part", "visible defect"]
BUSINESS_THEME_RULES["Competitor comparison"] += ["alternative brand", "may switch", "same feature"]


def rule_based_tags(text, rules=BUSINESS_THEME_RULES):
    text_norm = normalize_text(text)
    tags = []
    for label, patterns in rules.items():
        matched = False
        for pat in patterns:
            pat_norm = normalize_text(pat)
            if " " in pat_norm:
                matched = pat_norm in text_norm
            else:
                matched = re.search(rf"\b{re.escape(pat_norm)}\b", text_norm) is not None
            if matched:
                tags.append(label)
                break
    return tags

df["baseline_tags"] = df["clean_text"].apply(rule_based_tags)
df["baseline_tag_count"] = df["baseline_tags"].apply(len)

baseline_prevalence = (
    df["baseline_tags"]
    .explode()
    .value_counts()
    .rename("tagged_documents")
    .to_frame()
)
baseline_prevalence["prevalence"] = (baseline_prevalence["tagged_documents"] / len(df)).round(3)
display(baseline_prevalence)

hidden_label = df["primary_theme"].map(lambda x: THEME_BLUEPRINTS[x]["business_label"])
df["baseline_hits_primary"] = [label in tags for label, tags in zip(hidden_label, df["baseline_tags"])]
print(f"Baseline captured the hidden primary theme in {df['baseline_hits_primary'].mean():.1%} of documents.")

misses = df.loc[~df["baseline_hits_primary"], ["doc_id", "primary_theme", "baseline_tags", "text"]].head(6).copy()
misses["primary_theme"] = misses["primary_theme"].map(lambda x: THEME_BLUEPRINTS[x]["business_label"])
display(misses)

In [ ]:
# ============================================================
# How the unit of text changes prevalence under the same baseline
# ============================================================

sent_df["baseline_tags"] = sent_df["sentence_text"].apply(rule_based_tags)

doc_theme_mentions = df["baseline_tags"].explode().dropna().value_counts() / len(df)
sentence_theme_mentions = sent_df["baseline_tags"].explode().dropna().value_counts() / len(sent_df)
unit_compare = pd.concat([
    doc_theme_mentions.rename("document_unit_prevalence"),
    sentence_theme_mentions.rename("sentence_unit_prevalence"),
], axis=1).fillna(0).sort_values("document_unit_prevalence", ascending=False).round(3)

display(unit_compare)

print("Interpretation:")
print("Document-level prevalence means share of documents. Sentence-level prevalence means share of sentence chunks. Both are valid only if they match the decision unit.")

## 5. Representations for theme discovery

Count-based representations describe documents using visible words and phrases. Dense representations describe documents using lower-dimensional vectors that summarize similarity. Count-based representations are usually easier to explain. Dense representations can help when texts are short or when customers use different words for the same idea. In this notebook, dense vectors are created offline from TF-IDF plus SVD so that the exercise runs without external embedding models.

In [ ]:
# ============================================================
# 5. Representation choice: raw versus cleaned lexical representation
# ============================================================

raw_vectorizer = CountVectorizer(stop_words="english", ngram_range=(1, 2), min_df=4, max_df=0.85)
X_raw_counts = raw_vectorizer.fit_transform(df["raw_text"])
raw_terms = np.array(raw_vectorizer.get_feature_names_out())
raw_counts = np.asarray(X_raw_counts.sum(axis=0)).ravel()
raw_top = pd.DataFrame({
    "term": raw_terms[np.argsort(raw_counts)[::-1][:20]],
    "count": raw_counts[np.argsort(raw_counts)[::-1][:20]],
})

clean_vectorizer = CountVectorizer(stop_words="english", ngram_range=(1, 2), min_df=4, max_df=0.85)
X_clean_counts = clean_vectorizer.fit_transform(df["clean_text"])
clean_terms = np.array(clean_vectorizer.get_feature_names_out())
clean_counts = np.asarray(X_clean_counts.sum(axis=0)).ravel()
clean_top = pd.DataFrame({
    "term": clean_terms[np.argsort(clean_counts)[::-1][:20]],
    "count": clean_counts[np.argsort(clean_counts)[::-1][:20]],
})

print("Top terms before boilerplate removal")
display(raw_top)
print("Top terms after boilerplate removal")
display(clean_top)

print("The comparison shows why preprocessing policy is part of theme measurement, not a cosmetic step.")

In [ ]:
# ============================================================
# Sparse TF-IDF representation and dense SVD representation
# ============================================================

tfidf_vectorizer = TfidfVectorizer(
    stop_words="english",
    ngram_range=(1, 2),
    min_df=3,
    max_df=0.80,
    max_features=800,
)
X_tfidf = tfidf_vectorizer.fit_transform(df["clean_text"])
tfidf_terms = np.array(tfidf_vectorizer.get_feature_names_out())

svd_dim = min(30, X_tfidf.shape[1] - 1)
svd = TruncatedSVD(n_components=svd_dim, random_state=SEED)
X_dense = normalize(svd.fit_transform(X_tfidf))

print(f"Sparse TF-IDF shape: {X_tfidf.shape}")
print(f"Dense SVD representation shape: {X_dense.shape}")
print(f"Explained variance captured by {svd_dim} SVD dimensions: {svd.explained_variance_ratio_.sum():.2%}")

# Visualize the dense space in two dimensions for intuition.
pca = PCA(n_components=2, random_state=SEED)
coords = pca.fit_transform(X_dense)
coord_df = pd.DataFrame({
    "x": coords[:, 0],
    "y": coords[:, 1],
    "primary_theme": hidden_label,
    "channel": df["channel"],
})

fig, ax = plt.subplots(figsize=(7.5, 5.2))
for label in coord_df["primary_theme"].unique():
    subset = coord_df[coord_df["primary_theme"] == label]
    ax.scatter(subset["x"], subset["y"], s=22, alpha=0.65, label=label)
ax.set_title("Dense document map from TF-IDF plus SVD")
ax.set_xlabel("Dimension 1")
ax.set_ylabel("Dimension 2")
ax.legend(loc="center left", bbox_to_anchor=(1.02, 0.5), fontsize=8)
plt.tight_layout()
path = OUTPUT_DIR / "dense_document_map.png"
plt.savefig(path, dpi=160, bbox_inches="tight")
print(f"Saved figure: {path}")

## 6. Topic modeling with nonnegative matrix factorization

Nonnegative matrix factorization (NMF) is a useful baseline for business topic modeling because its outputs are easy to inspect. The model factorizes a document-term matrix into document-topic weights and topic-term weights. In managerial terms, the document-topic weights describe how much each theme appears in each document, and the topic-term weights describe the words and phrases that characterize each theme.

In [ ]:
# ============================================================
# 6. NMF topic modeling
# ============================================================

N_TOPICS = 7
nmf = NMF(n_components=N_TOPICS, init="nndsvda", random_state=SEED, max_iter=100)
W_nmf = nmf.fit_transform(X_tfidf)
H_nmf = nmf.components_

df["nmf_topic"] = W_nmf.argmax(axis=1)
df["nmf_weight"] = W_nmf.max(axis=1)

nmf_terms = component_terms_table(H_nmf, tfidf_terms, model_name="NMF", top_n=10)
display(nmf_terms)

print("Representative documents for each NMF topic")
for topic_id in range(N_TOPICS):
    print(f"\nNMF topic {topic_id}")
    reps = representative_docs(df, W_nmf, topic_id, score_name="nmf_weight", top_n=2)
    reps["text"] = reps["text"].apply(compact_text)
    display(reps)

In [ ]:
# ============================================================
# Topic resolution: fewer versus more themes
# ============================================================

resolution_rows = []
for k in [4, 6, 7, 10]:
    model = NMF(n_components=k, init="nndsvda", random_state=SEED, max_iter=100)
    W = model.fit_transform(X_tfidf)
    H = model.components_
    term_lists = top_terms_as_lists(H, tfidf_terms, top_n=8)
    coherence_scores = [topic_coherence_npmi(df["clean_text"], terms[:6]) for terms in term_lists]
    resolution_rows.append({
        "n_topics": k,
        "mean_npmi_screen": np.nanmean(coherence_scores),
        "mean_top_word_overlap": mean_pairwise_jaccard(term_lists),
        "mean_max_document_weight": float(W.max(axis=1).mean()),
    })

resolution_df = pd.DataFrame(resolution_rows).round(3)
display(resolution_df)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(resolution_df["n_topics"], resolution_df["mean_npmi_screen"], marker="o")
ax.set_title("NMF coherence screen across topic resolutions")
ax.set_xlabel("Number of topics")
ax.set_ylabel("Mean NPMI screen")
plt.tight_layout()
path = OUTPUT_DIR / "nmf_resolution_coherence.png"
plt.savefig(path, dpi=160, bbox_inches="tight")
print(f"Saved figure: {path}")

print("Use these numbers as screening evidence, not as automatic topic selection. The final choice should reflect actionability and stability.")

## 7. Probabilistic topic modeling with LDA

Latent Dirichlet Allocation (LDA) is a probabilistic topic model. It represents each topic as a probability distribution over words and each document as a probability distribution over topics. This is useful for customer text because a document can mention more than one issue. LDA usually works best when documents are long enough and vocabulary is consistent enough to support probabilistic word evidence.

In [ ]:
# ============================================================
# 7. LDA topic modeling
# ============================================================

lda_vectorizer = CountVectorizer(
    stop_words="english",
    ngram_range=(1, 2),
    min_df=3,
    max_df=0.80,
    max_features=800,
)
X_lda_counts = lda_vectorizer.fit_transform(df["clean_text"])
lda_terms = np.array(lda_vectorizer.get_feature_names_out())

lda = LatentDirichletAllocation(
    n_components=N_TOPICS,
    max_iter=10,
    learning_method="batch",
    random_state=SEED,
    doc_topic_prior=0.25,
    topic_word_prior=0.25,
)
Theta_lda = lda.fit_transform(X_lda_counts)

df["lda_topic"] = Theta_lda.argmax(axis=1)
df["lda_probability"] = Theta_lda.max(axis=1)

lda_terms_table = component_terms_table(lda.components_, lda_terms, model_name="LDA", top_n=10)
display(lda_terms_table)

mixture_strength = np.sort(Theta_lda, axis=1)[:, -2]
df["lda_second_probability"] = mixture_strength
mixed_docs = df.sort_values("lda_second_probability", ascending=False).head(8)[[
    "doc_id", "primary_theme", "secondary_theme", "lda_topic", "lda_probability", "lda_second_probability", "text"
]].copy()
mixed_docs["primary_theme"] = mixed_docs["primary_theme"].map(lambda x: THEME_BLUEPRINTS[x]["business_label"])
mixed_docs["secondary_theme"] = mixed_docs["secondary_theme"].map(lambda x: THEME_BLUEPRINTS[x]["business_label"] if x != "none" else "none")
mixed_docs["text"] = mixed_docs["text"].apply(compact_text)
display(mixed_docs)

print("LDA mixtures are useful when documents naturally contain more than one issue. The mixture still needs evidence review before becoming a business theme.")

## 8. Semantic-style topic discovery with dense vectors and clustering

Embedding-based topic discovery groups documents by meaning-oriented proximity and then explains the groups with representative documents and lexical descriptors. Production systems may use sentence transformers, dimensionality reduction, density-based clustering, and class-based TF-IDF. To keep this notebook offline and lightweight, we approximate the workflow with TF-IDF, SVD dense vectors, K-means clustering, and cluster-level descriptors.

In [ ]:
# ============================================================
# 8. Dense vectors + clustering + class-style TF-IDF descriptors
# ============================================================

kmeans = KMeans(n_clusters=N_TOPICS, random_state=SEED, n_init=8)
semantic_cluster = kmeans.fit_predict(X_dense)
df["semantic_cluster"] = semantic_cluster

# Class-style TF-IDF descriptors: aggregate documents by cluster, then score terms for each cluster.
cluster_texts = (
    df.assign(cluster=df["semantic_cluster"])
    .groupby("cluster")["clean_text"]
    .apply(lambda x: " ".join(x))
    .sort_index()
)
cluster_vectorizer = CountVectorizer(stop_words="english", ngram_range=(1, 2), min_df=1)
X_cluster_counts = cluster_vectorizer.fit_transform(cluster_texts)
cluster_feature_names = np.array(cluster_vectorizer.get_feature_names_out())
counts = X_cluster_counts.toarray().astype(float)
tf = counts / np.maximum(counts.sum(axis=1, keepdims=True), 1.0)
df_term = (counts > 0).sum(axis=0)
idf = np.log((1 + counts.shape[0]) / (1 + df_term)) + 1
ctfidf = tf * idf

cluster_descriptor_rows = []
for cluster_id in range(N_TOPICS):
    top = top_weighted_terms(ctfidf[cluster_id], cluster_feature_names, top_n=10)
    cluster_descriptor_rows.append({
        "cluster_id": cluster_id,
        "documents": int((semantic_cluster == cluster_id).sum()),
        "top_terms": ", ".join([term for term, weight in top]),
    })
cluster_descriptors = pd.DataFrame(cluster_descriptor_rows)
display(cluster_descriptors)

# Representative documents are closest to the cluster centroid in dense-vector space.
for cluster_id in range(N_TOPICS):
    members = np.where(semantic_cluster == cluster_id)[0]
    centroid = kmeans.cluster_centers_[cluster_id].reshape(1, -1)
    sims = cosine_similarity(X_dense[members], centroid).ravel()
    top_member_idx = members[np.argsort(sims)[::-1][:2]]
    reps = df.iloc[top_member_idx][["doc_id", "channel", "segment", "month_label", "primary_theme", "text"]].copy()
    reps["primary_theme"] = reps["primary_theme"].map(lambda x: THEME_BLUEPRINTS[x]["business_label"])
    reps["similarity_to_centroid"] = np.sort(sims)[::-1][:2].round(3)
    reps["text"] = reps["text"].apply(compact_text)
    print(f"\nSemantic-style cluster {cluster_id}")
    display(reps[["doc_id", "similarity_to_centroid", "channel", "segment", "month_label", "primary_theme", "text"]])

In [ ]:
# ============================================================
# Compare discovered grouping with hidden classroom labels
# ============================================================

hidden_codes = pd.Categorical(df["primary_theme"]).codes
nmf_ari = adjusted_rand_score(hidden_codes, df["nmf_topic"])
lda_ari = adjusted_rand_score(hidden_codes, df["lda_topic"])
semantic_ari = adjusted_rand_score(hidden_codes, df["semantic_cluster"])

comparison_scores = pd.DataFrame({
    "method": ["NMF on TF-IDF", "LDA on counts", "Dense SVD + K-means"],
    "adjusted_rand_against_hidden_labels": [nmf_ari, lda_ari, semantic_ari],
}).round(3)
display(comparison_scores)

print("These hidden labels are available only because this is synthetic classroom data. In real discovery, evaluation relies on evidence review, coherence, distinctness, stability, and decision usefulness.")

semantic_crosstab = pd.crosstab(
    df["primary_theme"].map(lambda x: THEME_BLUEPRINTS[x]["business_label"]),
    df["semantic_cluster"],
    normalize="index",
).round(2)
display(semantic_crosstab)

## 9. Validation: coherence, distinctness, stability, and theme cards

A topic model should not be treated as decision-ready simply because it produced clusters or word lists. Business validation asks whether the candidate themes are coherent, distinct, stable, and useful. Automated metrics can screen for problems, but they cannot replace domain review. A theme card records the evidence and naming rules that make future comparisons meaningful.

In [ ]:
# ============================================================
# 9. Lightweight validation screens for NMF topics
# ============================================================

nmf_term_lists = top_terms_as_lists(H_nmf, tfidf_terms, top_n=10)
lda_term_lists = top_terms_as_lists(lda.components_, lda_terms, top_n=10)
semantic_term_lists = []
for cluster_id in range(N_TOPICS):
    semantic_term_lists.append([term for term, weight in top_weighted_terms(ctfidf[cluster_id], cluster_feature_names, top_n=10)])

validation_rows = []
for method_name, term_lists in [
    ("NMF", nmf_term_lists),
    ("LDA", lda_term_lists),
    ("Dense cluster descriptors", semantic_term_lists),
]:
    coherence = [topic_coherence_npmi(df["clean_text"], terms[:6]) for terms in term_lists]
    validation_rows.append({
        "method": method_name,
        "mean_npmi_screen": np.nanmean(coherence),
        "mean_top_word_overlap": mean_pairwise_jaccard(term_lists),
        "lowest_topic_npmi_screen": np.nanmin(coherence),
    })

validation_df = pd.DataFrame(validation_rows).round(3)
display(validation_df)

# Stability check for NMF under two sampled refreshes.
def fit_sample_nmf_terms(texts, sample_frac=0.65, random_state=1, k=N_TOPICS, top_n=8):
    rng = np.random.default_rng(random_state)
    sample_idx = rng.choice(np.arange(len(texts)), size=int(len(texts) * sample_frac), replace=False)
    sampled_texts = pd.Series(texts).iloc[sample_idx]
    vec = TfidfVectorizer(stop_words="english", ngram_range=(1, 2), min_df=3, max_df=0.80, max_features=800)
    X = vec.fit_transform(sampled_texts)
    model = NMF(n_components=k, init="nndsvda", random_state=random_state, max_iter=100)
    model.fit_transform(X)
    return top_terms_as_lists(model.components_, np.array(vec.get_feature_names_out()), top_n=top_n)

terms_a = fit_sample_nmf_terms(df["clean_text"], random_state=7)
terms_b = fit_sample_nmf_terms(df["clean_text"], random_state=19)
stability = best_match_stability(terms_a, terms_b)
print(f"NMF top-word stability across two sampled refreshes: {stability:.3f}")

print("Interpretation guide:")
print("Higher NPMI suggests stronger word co-occurrence. Lower top-word overlap across topics suggests better distinctness. Higher sample stability suggests more durable theme definitions.")

In [ ]:
# ============================================================
# Theme naming and theme card generation
# ============================================================

OWNER_BY_LABEL = {blueprint["business_label"]: blueprint["owner"] for blueprint in THEME_BLUEPRINTS.values()}


def propose_theme_label(top_terms, rules=BUSINESS_THEME_RULES):
    term_text = " ".join(top_terms).lower()
    scores = {}
    for label, patterns in rules.items():
        score = 0
        for pat in patterns:
            pat_norm = normalize_text(pat)
            if pat_norm and pat_norm in term_text:
                score += 2 if " " in pat_norm else 1
        scores[label] = score
    best_label, best_score = max(scores.items(), key=lambda kv: kv[1])
    return best_label if best_score > 0 else "Needs human naming"

# Use NMF as the reporting catalog for this classroom example.
topic_to_label = {}
card_rows = []
for topic_id, terms in enumerate(nmf_term_lists):
    label = propose_theme_label(terms)
    topic_to_label[topic_id] = label
    reps = representative_docs(df, W_nmf, topic_id, score_name="nmf_weight", top_n=3)
    examples = " | ".join(reps["text"].apply(lambda x: compact_text(x, width=110)).tolist())
    card_rows.append({
        "theme_id": f"T{topic_id:02d}",
        "model_topic_id": topic_id,
        "working_name": label,
        "top_terms": ", ".join(terms),
        "inclusion_criteria": f"Text contains recurring evidence related to {label.lower()} and the issue would be interpreted similarly by a business reviewer.",
        "exclusion_criteria": "Exclude template-only process language, isolated mentions without a customer experience issue, and cases where another theme implies a different action owner.",
        "anchor_examples": examples,
        "likely_action_owner": OWNER_BY_LABEL.get(label, "Analytics lead and domain reviewer"),
        "review_cadence": "Monthly during discovery, quarterly after reporting definitions stabilize",
    })

theme_cards = pd.DataFrame(card_rows)
display(theme_cards)

theme_card_path = OUTPUT_DIR / "theme_cards_ch22.csv"
theme_cards.to_csv(theme_card_path, index=False)
print(f"Saved theme cards: {theme_card_path}")

## 10. Theme comparison across segments and time

Theme discovery becomes actionable when it supports comparisons. The most common business questions are where a theme is concentrated and how it is changing. Prevalence is usually the clearest default because it reports the share of documents in a group that mention a theme. Volume is useful when workload matters. Both measures must be interpreted with the same document unit, assignment rule, and corpus scope.

In [ ]:
# ============================================================
# 10. Theme prevalence by segment and time
# ============================================================

df["working_theme"] = df["nmf_topic"].map(topic_to_label)

# Topic assignment can create duplicate working names. Keep topic ID in the reporting label so comparisons are auditable.
df["reporting_theme"] = df.apply(lambda r: f"T{int(r['nmf_topic']):02d}: {r['working_theme']}", axis=1)

monthly_theme = (
    df.groupby(["month_label", "reporting_theme"])
    .size()
    .rename("volume")
    .reset_index()
)
monthly_total = df.groupby("month_label").size().rename("month_documents")
monthly_theme = monthly_theme.merge(monthly_total, on="month_label")
monthly_theme["prevalence"] = monthly_theme["volume"] / monthly_theme["month_documents"]

monthly_pivot = monthly_theme.pivot_table(index="month_label", columns="reporting_theme", values="prevalence", fill_value=0).sort_index()
first_half_mean = monthly_pivot.iloc[:6].mean()
latest = monthly_pivot.iloc[-1]
target_theme = (latest - first_half_mean).sort_values(ascending=False).index[0]

print(f"Theme selected for the heatmap because it increased most versus the first-half baseline: {target_theme}")

segment_theme = (
    df.groupby(["segment", "month_label", "reporting_theme"])
    .size()
    .rename("volume")
    .reset_index()
)
segment_total = df.groupby(["segment", "month_label"]).size().rename("group_documents")
segment_theme = segment_theme.merge(segment_total, on=["segment", "month_label"])
segment_theme["prevalence"] = segment_theme["volume"] / segment_theme["group_documents"]

heatmap_data = (
    segment_theme[segment_theme["reporting_theme"] == target_theme]
    .pivot_table(index="segment", columns="month_label", values="prevalence", fill_value=0)
    .reindex(index=SEGMENTS)
)
plot_heatmap(heatmap_data, f"Prevalence heatmap for {target_theme}", "theme_prevalence_heatmap.png")

prevalence_path = OUTPUT_DIR / "theme_prevalence_by_segment_month.csv"
segment_theme.to_csv(prevalence_path, index=False)
print(f"Saved prevalence table: {prevalence_path}")

In [ ]:
# ============================================================
# Representative evidence for the most prominent segment-by-time cell
# ============================================================

cell_values = heatmap_data.stack().rename("prevalence").reset_index()
cell_values = cell_values.sort_values("prevalence", ascending=False)
highlight = cell_values.iloc[0]
highlight_segment = highlight["segment"]
highlight_month = highlight["month_label"]

print(f"Highlighted cell: segment = {highlight_segment}, month = {highlight_month}, prevalence = {highlight['prevalence']:.1%}")

evidence = df[
    (df["segment"] == highlight_segment)
    & (df["month_label"] == highlight_month)
    & (df["reporting_theme"] == target_theme)
].sort_values("nmf_weight", ascending=False).head(5)[[
    "doc_id", "channel", "product_line", "region", "nmf_weight", "text"
]].copy()
evidence["text"] = evidence["text"].apply(compact_text)
display(evidence)

print("Measurement plus evidence is safer than a chart alone. The chart flags where to look, and the documents explain what the model is counting.")

## 11. Drift monitoring and governance outputs

Once themes enter dashboards, they become a measurement system. Monitoring should distinguish changes in customer language from changes in corpus composition, channel mix, templates, or theme definitions. The checks below are intentionally lightweight: prevalence alerts, channel-mix shifts, emerging vocabulary, and a versioned governance record.

In [ ]:
# ============================================================
# 11. Prevalence alerts and channel-mix audit
# ============================================================

# Alert when latest prevalence is much higher than the first six months.
baseline_period = monthly_pivot.index[:6]
latest_month = monthly_pivot.index[-1]
baseline_mean = monthly_pivot.loc[baseline_period].mean()
baseline_std = monthly_pivot.loc[baseline_period].std().replace(0, np.nan)
latest_prev = monthly_pivot.loc[latest_month]

alert_df = pd.DataFrame({
    "reporting_theme": monthly_pivot.columns,
    "baseline_mean_prevalence": baseline_mean.values,
    "baseline_std": baseline_std.values,
    "latest_month": latest_month,
    "latest_prevalence": latest_prev.values,
})
alert_df["z_like_shift"] = ((alert_df["latest_prevalence"] - alert_df["baseline_mean_prevalence"]) / alert_df["baseline_std"]).replace([np.inf, -np.inf], np.nan)
alert_df["alert_flag"] = (alert_df["z_like_shift"] > 2.0) & (alert_df["latest_prevalence"] >= 0.08)
alert_df = alert_df.sort_values("latest_prevalence", ascending=False).round(3)
display(alert_df)

alert_path = OUTPUT_DIR / "theme_prevalence_alerts.csv"
alert_df.to_csv(alert_path, index=False)
print(f"Saved alerts: {alert_path}")

# Channel mix audit: if corpus composition changed, theme prevalence may not be comparable.
channel_mix = pd.crosstab(df["month_label"], df["channel"], normalize="index").sort_index()
baseline_channel_mix = channel_mix.iloc[:6].mean()
latest_channel_mix = channel_mix.loc[latest_month]
channel_shift = (latest_channel_mix - baseline_channel_mix).sort_values(key=lambda s: s.abs(), ascending=False)
channel_shift_df = pd.DataFrame({
    "baseline_share": baseline_channel_mix,
    "latest_share": latest_channel_mix,
    "change": channel_shift,
}).round(3)
display(channel_shift_df)

print("A theme alert should be confirmed with evidence review and a corpus-mix audit before stakeholders act on it.")

In [ ]:
# ============================================================
# Emerging vocabulary and theme system governance record
# ============================================================

# Compare the last three months against earlier months to surface new or rising language.
cutoff_month = sorted(df["month"].unique())[-3]
early_texts = df[df["month"] < cutoff_month]["clean_text"]
late_texts = df[df["month"] >= cutoff_month]["clean_text"]

def count_terms(texts):
    counts = Counter()
    for text in texts:
        counts.update(simple_tokenize(text))
    return counts

early_counts = count_terms(early_texts)
late_counts = count_terms(late_texts)
rows = []
for term, late_count in late_counts.items():
    if late_count >= 3:
        early_count = early_counts.get(term, 0)
        lift = (late_count + 1) / (early_count + 1)
        if early_count == 0 or lift >= 2.5:
            rows.append({"term": term, "late_count": late_count, "early_count": early_count, "lift_ratio": lift})

vocab_drift = pd.DataFrame(rows).sort_values(["lift_ratio", "late_count"], ascending=False).head(20).round(2)
display(vocab_drift)

system_record = {
    "chapter": "Chapter 22 Topic Modeling and Theme Discovery",
    "corpus_definition": "Synthetic omnichannel customer feedback corpus for classroom practice",
    "unit_of_text": "One review, support ticket, survey verbatim, or community post as one document",
    "preprocessing_policy": {
        "case": "lowercase",
        "boilerplate": "remove known support templates",
        "ngrams": "unigrams and bigrams",
        "min_df": 3,
        "max_df": 0.80,
    },
    "reporting_model": "NMF on TF-IDF, 7 topics",
    "comparison_metric": "document prevalence within group and month",
    "validation_checks": ["NPMI screen", "top-word overlap", "sample stability", "representative document review"],
    "governance_policy": "Separate discovery refresh from reporting catalog updates. Version theme cards when names or boundaries change.",
}

record_path = OUTPUT_DIR / "topic_modeling_system_record.json"
with open(record_path, "w") as f:
    json.dump(system_record, f, indent=2)
print(f"Saved governance record: {record_path}")

print("System record preview")
print(json.dumps(system_record, indent=2))

## Decision guide

Use keyword and phrase baselines when the organization needs transparent measurement, fast diagnosis, or stable reporting categories. Use NMF when you want interpretable topic descriptors and document-topic weights from a lexical representation. Use LDA when mixture probabilities and probabilistic framing are useful, especially with longer and consistent documents. Use embedding-based clustering when short, informal, or paraphrased text makes lexical matching too brittle. In all cases, treat topics as candidate evidence. A business theme requires a name, definition, inclusion rules, exclusion rules, representative examples, validation checks, and a maintenance plan.

In [ ]:
# ============================================================
# Compact decision checklist as a classroom table
# ============================================================

decision_checklist = pd.DataFrame([
    {
        "decision_need": "Rapid diagnosis with known concerns",
        "recommended_start": "Keyword and phrase baseline",
        "evidence_to_review": "False matches, missed variants, prevalence by segment",
        "risk": "Rules can become too broad or stale",
    },
    {
        "decision_need": "Interpretable discovery from reviews or tickets",
        "recommended_start": "TF-IDF plus NMF",
        "evidence_to_review": "Top terms, high-weight documents, coherence, distinctness",
        "risk": "Paraphrases and boilerplate can fragment or distort topics",
    },
    {
        "decision_need": "Documents mix several issues",
        "recommended_start": "LDA or document-topic weights from NMF",
        "evidence_to_review": "Topic mixtures and representative mixed documents",
        "risk": "Probabilities can look more precise than the evidence supports",
    },
    {
        "decision_need": "Short, informal, or paraphrased text",
        "recommended_start": "Embeddings plus clustering, then lexical descriptors",
        "evidence_to_review": "Representative documents, cluster boundaries, outliers",
        "risk": "Grouping can be harder to explain without strong evidence trails",
    },
    {
        "decision_need": "Dashboard monitoring",
        "recommended_start": "Versioned theme catalog and prevalence metrics",
        "evidence_to_review": "Segment-time heatmaps, document examples, drift checks",
        "risk": "Theme names and boundaries can drift silently",
    },
])

display(decision_checklist)

## Exercises

**Exercise 1: Unit of text.** Split the corpus by sentence and fit an NMF model at the sentence level. Compare the top themes to the document-level themes. Which unit would you choose for a product fix dashboard, and why?

**Exercise 2: Baseline governance.** Add inclusion and exclusion rules for one business theme. Measure how many documents are added or removed by your rule changes. Review five changed examples and decide whether the new rule is better.

**Exercise 3: Topic resolution.** Fit NMF with 5, 7, and 12 topics. For each model, identify one theme that is too broad, one theme that is too narrow, and one theme that is useful for action.

**Exercise 4: LDA mixture interpretation.** Find three documents with high second-topic probability. Explain whether the mixture reflects a genuinely mixed customer experience or an ambiguous model boundary.

**Exercise 5: Segment and time comparison.** Choose one theme and build a segment-by-month prevalence heatmap. Add representative examples for the largest increase. State one business action and one alternative explanation.

**Exercise 6: Drift monitoring.** Add twenty future documents that mention a new feature or campaign. Rerun the vocabulary drift and prevalence alert sections. Decide whether to update the reporting catalog or keep the change in the discovery layer.

## Closing note

Topic modeling is valuable because it helps analysts read a corpus at scale. Its value does not come from replacing interpretation. Its value comes from organizing evidence so that interpretation becomes disciplined, repeatable, and connected to decisions. The analyst's responsibility is to make the measurement choices explicit: what counts as a document, what corpus is in scope, what representation defines similarity, how candidate topics are validated, and how theme definitions are maintained over time.